In [ ]:
from pyspark.sql.functions import *
catalog = dbutils.widgets.get("catalog")
df = spark.read.table(f"{catalog}.02_silver.jc_citibike")

In [5]:
df.printSchema()

root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_id: string (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_id: string (nullable = true)
 |-- start_lat: decimal(10,0) (nullable = true)
 |-- start_lng: decimal(10,0) (nullable = true)
 |-- end_lat: decimal(10,0) (nullable = true)
 |-- end_lng: decimal(10,0) (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- metadata: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)
 |-- trip_duration_min: double (nullable = true)
 |-- trip_start_date: date (nullable = true)



In [6]:
df = df.groupBy(col("trip_start_date"),col("start_station_name")).agg(
    round(avg(col("trip_duration_min")),2).alias("avg_trip_duration_in_mins"),
    count(col("ride_id")).alias("total_trips")
)

In [7]:
df.orderBy("trip_start_date").show()

+---------------+--------------------+-------------------------+-----------+
|trip_start_date|  start_station_name|avg_trip_duration_in_mins|total_trips|
+---------------+--------------------+-------------------------+-----------+
|     2025-02-28|   JC Medical Center|                     5.25|          1|
|     2025-02-28|    Grand St & 14 St|                     12.3|          1|
|     2025-02-28|Hoboken Terminal ...|                    25.25|          1|
|     2025-02-28|         Exchange Pl|                      2.8|          1|
|     2025-03-01|    Marin Light Rail|                     7.67|         41|
|     2025-03-01|      Jackson Square|                    33.43|          9|
|     2025-03-01|     McGinley Square|                     5.85|         16|
|     2025-03-01|    Heights Elevator|                    11.01|         11|
|     2025-03-01|      Riverview Park|                    11.94|         17|
|     2025-03-01|   Clinton St & 7 St|                     4.47|         29|

In [ ]:
df.write.mode("overwrite").option("overwriteSchema",True).saveAsTable(f"{catalog}.03_gold.daily_station_summary")